# 03: Probing Experiments — Main Results

**Goal**: Train probing classifiers on frozen embeddings and generate the core experimental results.

This notebook:
1. Loads cached embeddings from all models
2. Trains logistic regression probes for each relation type
3. Reports F1 scores per model × relation
4. Creates the main results comparison table
5. Analyzes which embedding dimensions are diagnostic per relation

All experimental design principles from CLAUDE.md:
- Stratified k-fold CV (k=5) to handle class imbalance
- Binary probing per relation (cleaner signal than multiclass)
- Report macro F1 (class-weighted) not just accuracy
- Keep regularization weak (high C) to test representation, not regularize it away

## 1. Setup and Load Cached Embeddings

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
import os

# Notebooks run from notebooks/ — resolve paths relative to project root
PROJECT_ROOT = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()).resolve()
sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / ".env")

CACHE_DIR = Path(os.getenv("CACHE_DIR", "results/embeddings"))
if not CACHE_DIR.is_absolute():
    CACHE_DIR = PROJECT_ROOT / CACHE_DIR
RESULTS_DIR = PROJECT_ROOT / "results"

np.random.seed(42)
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 8)

print(f"Project root: {PROJECT_ROOT}")
print(f"Cache directory: {CACHE_DIR}")
print(f"Available embeddings:")
for f in sorted(CACHE_DIR.glob("*.npy")):
    size_mb = f.stat().st_size / (1024*1024)
    print(f"  {f.name:30s} {size_mb:8.1f} MB")

## 2. Load Embeddings and Dataset Labels

In [ ]:
from datasets import load_dataset

# Load embeddings (multimodal added later once image extraction is complete)
print("Loading cached embeddings...")
sbert_embeddings = np.load(CACHE_DIR / "sbert_vsr_train.npy")
clip_text_embeddings = np.load(CACHE_DIR / "clip_text_vsr_train.npy")

print(f"Embedding shapes:")
print(f"  SBERT:        {sbert_embeddings.shape}")
print(f"  CLIP text:    {clip_text_embeddings.shape}")

# Load VSR dataset for relation types and labels
print(f"\nLoading VSR dataset...")
vsr = load_dataset("cambridgeltl/vsr_random")
train_data = vsr["train"]

relation_types = np.array([ex["relation"] for ex in train_data])
binary_labels = np.array([int(ex["label"]) for ex in train_data])

print(f"Dataset: {len(train_data)} examples")
print(f"Relation types: {len(np.unique(relation_types))} unique")
print(f"Label dist: {np.sum(binary_labels)} True, {len(binary_labels) - np.sum(binary_labels)} False")

## 3. Train Probes for Each Model

In [ ]:
from src.probing import probe_by_relation_type, model_comparison_table

print("Training SBERT probe...")
sbert_results = probe_by_relation_type(
    sbert_embeddings, relation_types, binary_labels, n_splits=5, C=1.0, min_samples=20
)
print(f"  Relations tested: {len(sbert_results)}")
print(f"  Mean F1: {sbert_results['f1_macro_mean'].mean():.3f}")

print(f"\nTraining CLIP text probe...")
clip_text_results = probe_by_relation_type(
    clip_text_embeddings, relation_types, binary_labels, n_splits=5, C=1.0, min_samples=20
)
print(f"  Relations tested: {len(clip_text_results)}")
print(f"  Mean F1: {clip_text_results['f1_macro_mean'].mean():.3f}")

## 4. Main Results Table

In [ ]:
results_dict = {
    "sbert": sbert_results,
    "clip_text": clip_text_results,
}

comparison = model_comparison_table(results_dict)
print(comparison.head(15).to_string())

comparison.to_csv(RESULTS_DIR / "probing_results.csv", index=False)
print(f"\nSaved results table to results/probing_results.csv")

## 5. Summary Statistics

In [ ]:
print("\n" + "="*70)
print("PROBING RESULTS SUMMARY")
print("="*70)

for model_name, results_df in results_dict.items():
    f1_scores = results_df["f1_macro_mean"].values
    print(f"\n{model_name.upper():15s}:")
    print(f"  Mean F1:       {f1_scores.mean():.3f} (±{f1_scores.std():.3f})")
    print(f"  Median F1:     {np.median(f1_scores):.3f}")
    print(f"  Min F1:        {f1_scores.min():.3f}")
    print(f"  Max F1:        {f1_scores.max():.3f}")
    print(f"  Relations:     {len(results_df)}")

print(f"\nCLIP text vs SBERT:")
common = set(sbert_results["relation"]) & set(clip_text_results["relation"])
sbert_f1   = {r: f for r, f in zip(sbert_results["relation"], sbert_results["f1_macro_mean"])}
clip_f1    = {r: f for r, f in zip(clip_text_results["relation"], clip_text_results["f1_macro_mean"])}
gains = [clip_f1[r] - sbert_f1[r] for r in common]
print(f"  Mean gain: {np.mean(gains):+.3f} (±{np.std(gains):.3f})")
print(f"  Relations where CLIP wins: {sum(g > 0 for g in gains)}/{len(gains)}")
print("\n" + "="*70)

## 6. Visualize Results as Heatmap

In [ ]:
model_names = list(results_dict.keys())
f1_cols = [f"{m}_f1" for m in model_names]

heatmap_data = comparison[f1_cols].copy()
heatmap_data.index = comparison["relation"]
heatmap_data.columns = [m.upper().replace("_", " ") for m in model_names]

top_n = 20
top_relations = heatmap_data.mean(axis=1).nlargest(top_n).index
heatmap_data = heatmap_data.loc[top_relations]

fig, ax = plt.subplots(figsize=(7, 10))
sns.heatmap(
    heatmap_data,
    annot=True, fmt=".3f",
    cmap="RdYlGn", vmin=0, vmax=1,
    cbar_kws={"label": "F1 Score"},
    ax=ax, linewidths=0.5,
)
ax.set_title(f"Probing Results: Top {top_n} Relations (by mean F1)", fontsize=13, fontweight="bold")
ax.set_xlabel("Model", fontsize=11)
ax.set_ylabel("Relation Type", fontsize=11)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "figures" / "03_results_heatmap.png", dpi=300, bbox_inches="tight")
plt.show()
print("Saved: 03_results_heatmap.png")

## 7. Detailed Relation Analysis

In [ ]:
comparison_copy = comparison.copy()
comparison_copy["clip_text_gain"] = comparison_copy["clip_text_f1"] - comparison_copy["sbert_f1"]

print("Relations with LARGEST CLIP TEXT gains over SBERT:")
top_gains = comparison_copy.nlargest(5, "clip_text_gain")[["relation", "sbert_f1", "clip_text_f1", "clip_text_gain"]]
for _, row in top_gains.iterrows():
    print(f"  {row['relation']:25s}: {row['sbert_f1']:.3f} → {row['clip_text_f1']:.3f} ({row['clip_text_gain']:+.3f})")

print(f"\nRelations where SBERT outperforms CLIP text:")
sbert_better = comparison_copy[comparison_copy["sbert_f1"] > comparison_copy["clip_text_f1"]]
if len(sbert_better) > 0:
    for _, row in sbert_better.head(5).iterrows():
        print(f"  {row['relation']:25s}: SBERT {row['sbert_f1']:.3f} vs CLIP {row['clip_text_f1']:.3f} ({row['clip_text_gain']:+.3f})")
else:
    print("  (None — CLIP text superior across all relations)")

## 8. Save Detailed Results

## 9. Probe Weight Analysis

This is one of the most interpretability-relevant outputs of the study.

After fitting a multiclass probe across all relation types jointly, we inspect the learned coefficients (`probe.coef_`, shape `[n_relations, embedding_dim]`) to ask: **which embedding dimensions are most diagnostic for each spatial relation?**

This is behavioral interpretability in the tradition of Dalvi et al. (2019) "What Is One Grain of Sand in the Desert?" — we're not asking *how* CLIP computes spatial structure, but *where* that structure lives in the embedding geometry.

Key questions this analysis can answer:
- Do `above` and `below` use the same dimensions with opposite signs? (If so, the model has a vertical axis)
- Do `left of` and `right of` share dimensions? (If not, the model may lack a consistent horizontal axis)
- Are CLIP's diagnostic dimensions different from SBERT's? (Suggesting different geometric structures)

Caveats: this is a linear probe on frozen embeddings, not a mechanistic account. High-weight dimensions correlate with, but do not causally explain, the model's spatial encoding.

In [ ]:
from src.probing import probe_weight_analysis
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder

# Encode relation types as integers for multiclass probe
le = LabelEncoder()
relation_encoded = le.fit_transform(relation_types)

weight_results = {}

for model_name, embeddings in [("sbert", sbert_embeddings), ("clip_text", clip_text_embeddings)]:
    print(f"\n--- {model_name.upper()} probe weight analysis ---")
    coef, top_dims = probe_weight_analysis(embeddings, relation_encoded, top_k=10)
    weight_results[model_name] = {"coef": coef, "top_dims": top_dims}

    print(f"  Coefficient matrix shape: {coef.shape}  (n_relations × embedding_dim)")
    print(f"  Top 10 most diagnostic dimensions (by max weight magnitude across relations):")
    for dim_name, weight in top_dims:
        print(f"    {dim_name:8s}: {weight:+.4f}")

# Check if above/below share dimensions with opposite signs (vertical axis hypothesis)
print("\n--- Vertical axis check: 'above' vs 'below' ---")
for model_name, embeddings in [("sbert", sbert_embeddings), ("clip_text", clip_text_embeddings)]:
    clf = LogisticRegression(C=1.0, max_iter=1000, solver="lbfgs", random_state=42, multi_class="multinomial")
    clf.fit(embeddings, relation_encoded)
    classes = list(le.classes_)

    if "above" in classes and "below" in classes:
        above_idx = classes.index("above")
        below_idx = classes.index("below")
        above_coef = clf.coef_[above_idx]
        below_coef = clf.coef_[below_idx]
        correlation = np.corrcoef(above_coef, below_coef)[0, 1]
        print(f"  {model_name.upper():12s}: above/below weight correlation = {correlation:+.3f}")
        print(f"    (negative = opposite-sign dimensions = vertical axis present)")
    else:
        print(f"  {model_name.upper():12s}: 'above' or 'below' not found in relations")

In [ ]:
for model_name, results_df in results_dict.items():
    csv_path = RESULTS_DIR / f"probing_results_{model_name}_detailed.csv"
    results_df.to_csv(csv_path, index=False)
    print(f"Saved: {csv_path.name}")

comparison_copy.to_csv(RESULTS_DIR / "probing_results_with_gains.csv", index=False)
print(f"Saved: probing_results_with_gains.csv")

## 9. Key Findings

In [ ]:
print("\n" + "="*70)
print("KEY FINDINGS")
print("="*70)

print(f"\n1. OVERALL PERFORMANCE:")
for model_name in model_names:
    f1 = comparison[f"{model_name}_f1"].mean()
    print(f"   {model_name.upper():20s}: {f1:.1%} mean F1")

print(f"\n2. VISUAL GROUNDING BENEFIT (CLIP text vs SBERT):")
gain = comparison["clip_text_f1"].mean() - comparison["sbert_f1"].mean()
print(f"   Mean F1 gain: {gain:+.1%}")
print(f"   Relations where CLIP wins: {(comparison['clip_text_f1'] > comparison['sbert_f1']).sum()}/{len(comparison)}")

print(f"\n3. HARDEST RELATIONS (lowest CLIP text F1):")
for _, row in comparison.nsmallest(3, "clip_text_f1").iterrows():
    print(f"   {row['relation']:25s}: {row['clip_text_f1']:.3f}")

print(f"\n4. EASIEST RELATIONS (highest CLIP text F1):")
for _, row in comparison.nlargest(3, "clip_text_f1").iterrows():
    print(f"   {row['relation']:25s}: {row['clip_text_f1']:.3f}")

print("\n" + "="*70)